# Notebook 05: Episode Index (XuetangX)

**Purpose:** Build episode index — K=5 support pairs, Q=10 query pairs per user.

**Protocol:** Support timestamps < query timestamps (chronological, no leakage).

**Expected episode counts (new pipeline, 70/15/15 split):**
```python
assert len(episodes_train) == 113920
assert len(episodes_val)   == 942
assert len(episodes_test)  == 986
```

**Inputs:** `data/processed/xuetangx/pairs/pairs_{train|val|test}.parquet`

**Outputs:** `data/processed/xuetangx/episodes/episodes_{train|val|test}_K5_Q10.parquet`

In [1]:
# [CELL 05-00] Bootstrap

import json, time, uuid, hashlib
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List
import numpy as np, pandas as pd
import duckdb

def find_repo_root(start):
    for p in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (p / 'PROJECT_STATE.md').exists(): return p
    raise RuntimeError('PROJECT_STATE.md not found')

REPO_ROOT = find_repo_root(Path.cwd())
PATHS = {
    'META_REGISTRY':  REPO_ROOT / 'meta.json',
    'DATA_INTERIM':   REPO_ROOT / 'data' / 'interim',
    'DATA_PROCESSED': REPO_ROOT / 'data' / 'processed',
    'REPORTS':        REPO_ROOT / 'reports',
}

def cell_start(cid, title, **kw):
    t = time.time()
    print(f'\n[{cid}] {title}')
    for k,v in kw.items(): print(f'[{cid}] {k}={v}')
    return t

def cell_end(cid, t0, **kw):
    for k,v in kw.items(): print(f'[{cid}] {k}={v}')
    print(f'[{cid}] elapsed={time.time()-t0:.2f}s  done')

def write_json_atomic(path, obj, indent=2):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f'.tmp_{uuid.uuid4().hex}')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=indent), encoding='utf-8')
    tmp.replace(path)

def read_json(path): return json.loads(Path(path).read_text(encoding='utf-8'))
print(f'[CELL 05-00] REPO_ROOT={REPO_ROOT}  done')

[CELL 05-00] REPO_ROOT=/home/user/jamalla/anonymous-users-mooc-session-meta  done


In [2]:
# [CELL 05-01] Seed + config

import random
GLOBAL_SEED = 20260107
random.seed(GLOBAL_SEED); np.random.seed(GLOBAL_SEED)

NOTEBOOK_NAME = '05_episode_index_xuetangx'
DATASET       = 'xuetangx'
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ID  = uuid.uuid4().hex

OUT_DIR = PATHS['REPORTS'] / NOTEBOOK_NAME / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH   = OUT_DIR / 'report.json'
CONFIG_PATH   = OUT_DIR / 'config.json'
MANIFEST_PATH = OUT_DIR / 'manifest.json'

DUCKDB_PATH = PATHS['DATA_INTERIM'] / 'xuetangx.duckdb'
PAIRS_DIR   = PATHS['DATA_PROCESSED'] / 'xuetangx' / 'pairs'
EPS_DIR     = PATHS['DATA_PROCESSED'] / 'xuetangx' / 'episodes'
EPS_DIR.mkdir(parents=True, exist_ok=True)

PAIRS_TRAIN = PAIRS_DIR / 'pairs_train.parquet'
PAIRS_VAL   = PAIRS_DIR / 'pairs_val.parquet'
PAIRS_TEST  = PAIRS_DIR / 'pairs_test.parquet'

K_SUPPORT = 5
Q_QUERY   = 10

OUT_EPS_TRAIN = EPS_DIR / f'episodes_train_K{K_SUPPORT}_Q{Q_QUERY}.parquet'
OUT_EPS_VAL   = EPS_DIR / f'episodes_val_K{K_SUPPORT}_Q{Q_QUERY}.parquet'
OUT_EPS_TEST  = EPS_DIR / f'episodes_test_K{K_SUPPORT}_Q{Q_QUERY}.parquet'

CFG = {'notebook': NOTEBOOK_NAME, 'dataset': DATASET, 'seed': GLOBAL_SEED,
       'K_support': K_SUPPORT, 'Q_query': Q_QUERY}
write_json_atomic(CONFIG_PATH, CFG)
for path, obj in [(REPORT_PATH, {'run_id':RUN_ID,'notebook':NOTEBOOK_NAME,'run_tag':RUN_TAG,
                                   'created_at':datetime.now().isoformat(timespec='seconds'),
                                   'metrics':{},'key_findings':[],'sanity_samples':{},'data_fingerprints':{},'notes':[]}),
                   (MANIFEST_PATH, {'run_id':RUN_ID,'notebook':NOTEBOOK_NAME,'run_tag':RUN_TAG,'artifacts':[]})]: 
    write_json_atomic(path, obj)
META = PATHS['META_REGISTRY']
if not META.exists(): write_json_atomic(META, {'schema_version':1,'runs':[]})
m = read_json(META); m['runs'].append({'run_id':RUN_ID,'notebook':NOTEBOOK_NAME,'run_tag':RUN_TAG,'out_dir':str(OUT_DIR)})
write_json_atomic(META, m)
print(f'[CELL 05-01] K={K_SUPPORT}  Q={Q_QUERY}  run={RUN_TAG}  done')

[CELL 05-01] K=5  Q=10  run=20260408_142504  done


In [3]:
# [CELL 05-02] Load split pairs

t0 = cell_start('CELL 05-02', 'Load split pairs')

for p in [PAIRS_TRAIN, PAIRS_VAL, PAIRS_TEST]:
    if not p.exists(): raise RuntimeError(f'Missing {p}. Run 04_user_split_xuetangx.ipynb first.')

pairs_train = pd.read_parquet(PAIRS_TRAIN)
pairs_val   = pd.read_parquet(PAIRS_VAL)
pairs_test  = pd.read_parquet(PAIRS_TEST)

for name, df in [('train', pairs_train), ('val', pairs_val), ('test', pairs_test)]:
    print(f'[CELL 05-02] {name}: {len(df):,} pairs, {df["user_id"].nunique():,} users')

cell_end('CELL 05-02', t0)


[CELL 05-02] Load split pairs
[CELL 05-02] train: 344,532 pairs, 49,566 users
[CELL 05-02] val: 69,663 pairs, 10,621 users
[CELL 05-02] test: 73,501 pairs, 10,622 users
[CELL 05-02] elapsed=0.32s  done


In [4]:
# [CELL 05-03] Episode creation function

t0 = cell_start('CELL 05-03', 'Define episode creation function')

def create_episodes(pairs_df: pd.DataFrame, K: int, Q: int, mode: str) -> List[Dict[str, Any]]:
    """
    Build episodic meta-learning index.
    Each user forms one episode (val/test) or multiple episodes via sliding window (train).
    Support timestamps < query timestamps — no leakage.
    """
    MIN = K + Q
    episodes = []
    eid = 0

    for user_id, grp in pairs_df.groupby('user_id'):
        grp = grp.sort_values('label_ts_epoch').reset_index(drop=True)
        n = len(grp)
        if n < MIN: continue

        if mode in ('val', 'test'):
            # Single episode: last K+Q pairs (chronological)
            window = grp.tail(MIN)
            sup = window.iloc[:K]
            qry = window.iloc[K:]
            episodes.append({
                'episode_id': eid,
                'user_id': user_id,
                'support_pair_ids': sup['pair_id'].tolist(),
                'query_pair_ids':   qry['pair_id'].tolist(),
                'support_max_ts': int(sup['label_ts_epoch'].max()),
                'query_min_ts':   int(qry['label_ts_epoch'].min()),
            })
            eid += 1
        else:
            # Train: sliding window over all valid positions
            for start in range(0, n - MIN + 1):
                window = grp.iloc[start:start+MIN]
                sup = window.iloc[:K]
                qry = window.iloc[K:]
                # Chronological guard
                if sup['label_ts_epoch'].max() >= qry['label_ts_epoch'].min(): continue
                episodes.append({
                    'episode_id': eid,
                    'user_id': user_id,
                    'support_pair_ids': sup['pair_id'].tolist(),
                    'query_pair_ids':   qry['pair_id'].tolist(),
                    'support_max_ts': int(sup['label_ts_epoch'].max()),
                    'query_min_ts':   int(qry['label_ts_epoch'].min()),
                })
                eid += 1

    return episodes

print('[CELL 05-03] Episode function defined')
cell_end('CELL 05-03', t0)


[CELL 05-03] Define episode creation function
[CELL 05-03] Episode function defined
[CELL 05-03] elapsed=0.00s  done


In [5]:
# [CELL 05-04] Build and save episodes

t0 = cell_start('CELL 05-04', 'Build episodes K=5 Q=10')

episodes_train = pd.DataFrame(create_episodes(pairs_train, K_SUPPORT, Q_QUERY, 'train'))
episodes_val   = pd.DataFrame(create_episodes(pairs_val,   K_SUPPORT, Q_QUERY, 'val'))
episodes_test  = pd.DataFrame(create_episodes(pairs_test,  K_SUPPORT, Q_QUERY, 'test'))

for name, df in [('train', episodes_train), ('val', episodes_val), ('test', episodes_test)]:
    print(f'[CELL 05-04] {name}: {len(df):,} episodes, {df["user_id"].nunique():,} users')

# ── ASSERTIONS (new pipeline: 70/15/15 split) ──
assert len(episodes_train) == 113920, f'Expected 113920 train episodes, got {len(episodes_train)}'
assert len(episodes_val)   == 942,    f'Expected 942 val episodes, got {len(episodes_val)}'
assert len(episodes_test)  == 986,    f'Expected 986 test episodes, got {len(episodes_test)}'
print('[CELL 05-04] Episode count assertions PASSED')

cell_end('CELL 05-04', t0)


[CELL 05-04] Build episodes K=5 Q=10
[CELL 05-04] train: 113,920 episodes, 4,645 users
[CELL 05-04] val: 942 episodes, 942 users
[CELL 05-04] test: 986 episodes, 986 users
[CELL 05-04] Episode count assertions PASSED
[CELL 05-04] elapsed=29.68s  done


In [6]:
# [CELL 05-05] Validate chronological ordering

t0 = cell_start('CELL 05-05', 'Validate chronological ordering')

for name, df in [('train', episodes_train), ('val', episodes_val), ('test', episodes_test)]:
    violations = df[df['support_max_ts'] >= df['query_min_ts']]
    if len(violations) > 0:
        raise RuntimeError(f'{name}: {len(violations)} chronological violations!')
    print(f'[CELL 05-05] {name}: all {len(df):,} episodes chronologically valid')

cell_end('CELL 05-05', t0)


[CELL 05-05] Validate chronological ordering
[CELL 05-05] train: all 113,920 episodes chronologically valid
[CELL 05-05] val: all 942 episodes chronologically valid
[CELL 05-05] test: all 986 episodes chronologically valid
[CELL 05-05] elapsed=0.00s  done


In [7]:
# [CELL 05-06] Save episodes + register DuckDB views

t0 = cell_start('CELL 05-06', 'Save + register DuckDB views')

episodes_train.to_parquet(OUT_EPS_TRAIN, index=False, compression='zstd')
episodes_val.to_parquet(OUT_EPS_VAL,   index=False, compression='zstd')
episodes_test.to_parquet(OUT_EPS_TEST,  index=False, compression='zstd')

def esc(p): return str(p).replace("'","''")
con = duckdb.connect(str(DUCKDB_PATH), read_only=False)
for split, path in [('train', OUT_EPS_TRAIN), ('val', OUT_EPS_VAL), ('test', OUT_EPS_TEST)]:
    vname = f'xuetangx_episodes_{split}_K{K_SUPPORT}_Q{Q_QUERY}'
    con.execute(f'DROP VIEW IF EXISTS {vname};')
    con.execute(f"CREATE VIEW {vname} AS SELECT * FROM read_parquet('{esc(path)}')")
    n = con.execute(f'SELECT COUNT(*) FROM {vname}').fetchone()[0]
    print(f'[CELL 05-06] {vname}: {n:,}')
con.close()

# Report
r = read_json(REPORT_PATH)
r['metrics'] = {'K': K_SUPPORT, 'Q': Q_QUERY,
                 'n_episodes_train': len(episodes_train),
                 'n_episodes_val':   len(episodes_val),
                 'n_episodes_test':  len(episodes_test)}
write_json_atomic(REPORT_PATH, r)

cell_end('CELL 05-06', t0)


[CELL 05-06] Save + register DuckDB views
[CELL 05-06] xuetangx_episodes_train_K5_Q10: 113,920
[CELL 05-06] xuetangx_episodes_val_K5_Q10: 942
[CELL 05-06] xuetangx_episodes_test_K5_Q10: 986
[CELL 05-06] elapsed=0.60s  done


## Notebook 05 Complete

Episodes built (K=5, Q=10). Chronological ordering validated.

**Next:** `06_base_model_selection_xuetangx.ipynb` → produces `gru_global_xuetangx.pth`.